## 1. Setup and settings
Place `network.graphml` and `critical_sets.csv` in `PROJECT_DIR`.

In [1]:
%pip -qq install osmnx networkx pandas numpy scikit-learn qiskit qiskit-aer scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.4 MB/s eta 0:00:00


In [2]:
import json
from pathlib import Path
from ast import literal_eval
from itertools import combinations, product
from math import fsum, ceil
from time import perf_counter
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
from scipy.optimize import minimize
from scipy.stats import binomtest
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator
from qiskit_aer.library import save_amplitudes_squared
from IPython.display import display

In [3]:
# Change your environment paths if not working in Colab

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/quantum_road_scheduling")
CACHE_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR = PROJECT_DIR / "demo_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [4]:
SEED = 666                  # for reproducibility
DEPTH = 1                              # Fixed p=1 model.
PENALTY = 2.0                          # QUBO lambda: overlapping-pair penalty.
MIN_FEASIBILITY = 0.01      # fraction of feasible samples tolerated
CVAR_FRACTION = 0.10
GRID_SIZE = 31                         # Search 31 x 31 analytical grid.
SHORTLIST_SIZE = 8
GRID_SEPARATION = 3.0                  # distance in grid-cell units.
MAX_OPT_EVALS = 80
MAX_SIMULATOR_CALLS = 100
RHOBEG = 0.01
OPT_TOL = 1e-4
CVAR_TIE_TOL = 1e-9
FINAL_SHOTS = 3_200_000     # noiseless Aer sampling for trained QAOA
BATCH_SHOTS = 200_000
FINAL_STREAM = 200
CHUNK_SIZE = 16
DETOUR_TIE_TOL_S = 1e-6
RUN_TRAINING = True         # turn False after first run
RUN_SAMPLING = True         # turn False after first run

PARAMETER_FILE = None

ORIGIN = (45.40917802385951, -73.65906344074625)    # bridge
DESTINATION = (45.34599185120908, -73.7638231)      # hospital

## 2. Geography and feasible schedules
Use the supplied critical relationships and set of jobs
Cache the baseline route and 35 closure times. A selected pair forms a
two-job period; unmatched jobs form singleton periods. Detour breaks might be used for final ranking ties only (WIP).

In [5]:
units = {
    "B": (128919678, 139880600), "C": (131102891, 266719341),
    "D": (137980893, 10295557258), "E": (3495233768, 7939009570),
    "F": (266719341, 10096380232), "G": (296996256, 7227651553),
    "H": (8084937097, 9814565499), "I": (13324554804, 13510598399),
    "J": (3495233768, 13510598398), "K": (13510598398, 13510598399),
}
for filename in ["network.graphml", "critical_sets.csv"]:
    if not (PROJECT_DIR / filename).exists():
        raise FileNotFoundError(f"Place {filename} in {PROJECT_DIR}.")
labels = tuple(sorted(units))
critical = pd.read_csv(PROJECT_DIR / "critical_sets.csv")
critical_pairs = {
    tuple(sorted((tuple(sorted(literal_eval(row.unit_1))),
                  tuple(sorted(literal_eval(row.unit_2))))))
    for row in critical.loc[critical["size"].eq(2)].itertuples(index=False)
}
compatible_pairs = sorted(
    a + b for a, b in combinations(labels, 2)
    if tuple(sorted((units[a], units[b]))) not in critical_pairs
)
pair_to_qubit = {pair: q for q, pair in enumerate(compatible_pairs)}

G = ox.io.load_graphml(PROJECT_DIR / "network.graphml")
G_route = ox.truncate.largest_component(G, strongly=True)
origin_node = ox.distance.nearest_nodes(G_route, X=ORIGIN[1], Y=ORIGIN[0])
destination_node = ox.distance.nearest_nodes(G_route, X=DESTINATION[1], Y=DESTINATION[0])
closure_file = OUTPUT_DIR / "closure_times.csv"
cache_file = closure_file if closure_file.exists() else CACHE_DIR / "closure_times.csv"
closures = pd.read_csv(cache_file) if cache_file.exists() else None

if closures is not None and "baseline_route_nodes" in closures:
    baseline_route = json.loads(closures["baseline_route_nodes"].dropna().iloc[0])
else:
    baseline_route = nx.shortest_path(G_route, origin_node, destination_node, weight="travel_time")
if closures is not None:
    baseline_time_s = float(closures.iloc[0]["route_time_s"] - closures.iloc[0]["excess_time_s"])
else:
    baseline_time_s = fsum(
        min(edge["travel_time"] for edge in G_route[u][v].values())
        for u, v in zip(baseline_route, baseline_route[1:])
    )

    def closure_time(batch):
        closed = G_route.copy()
        for label in batch:
            u, v = units[label]
            for a, b in ((u, v), (v, u)):
                if closed.has_edge(a, b):
                    closed.remove_edges_from([(a, b, key) for key in list(closed[a][b])])
        return float(nx.shortest_path_length(closed, origin_node, destination_node, weight="travel_time"))

    rows = []
    for batch in list(labels) + compatible_pairs:
        route_time_s = closure_time(batch)
        rows.append({"batch": batch, "size": len(batch), "qubit": pair_to_qubit.get(batch),
                     "route_time_s": route_time_s, "excess_time_s": route_time_s - baseline_time_s})
    closures = pd.DataFrame(rows)

# Store baseline metadata in the same reusable cache, without a separate file.
closures["baseline_time_s"] = baseline_time_s
closures["baseline_route_nodes"] = pd.Series([None] * len(closures), dtype="object")
closures.loc[0, "baseline_route_nodes"] = json.dumps([int(node) for node in baseline_route])
closures.to_csv(closure_file, index=False)
batch_time = closures.set_index("batch")["route_time_s"].to_dict()

In [6]:
def enumerate_matchings(remaining, mask=0, pairs=(), singles=()):
    if not remaining:
        yield mask, pairs, singles
        return
    first, *rest = remaining
    yield from enumerate_matchings(rest, mask, pairs, singles + (first,))
    for partner in rest:
        pair = first + partner
        if pair in pair_to_qubit:
            yield from enumerate_matchings(
                [job for job in rest if job != partner],
                mask | (1 << pair_to_qubit[pair]), pairs + (pair,), singles,
            )

rows = []
for mask, pairs, singles in enumerate_matchings(labels):
    batches = sorted(pairs + singles)
    total_time = fsum(batch_time[batch] for batch in batches)
    rows.append({"mask": mask, "pairs": " ".join(pairs), "singletons": " ".join(singles),
                 "batches": " ".join(batches), "periods": len(batches),
                 "total_route_time_s": total_time,
                 "total_excess_time_s": total_time - len(batches) * baseline_time_s})
schedules = pd.DataFrame(rows).sort_values("mask").reset_index(drop=True)
schedules.to_csv(OUTPUT_DIR / "schedules.csv", index=False)

## 3. QAOA model
For the 25 compatible-pair variables, use
\(E(x)=-\sum_a x_a+2\sum_{(a,b){overlapping}}x_ax_b\).
The p=1 circuit applies cost then X-mixer rotations, with explicit order
`[gamma[0], beta[0]]`. Route times do not enter this model.

In [7]:
n_qubits = len(compatible_pairs)
overlaps = [(a, b) for a, b in combinations(range(n_qubits), 2)
            if set(compatible_pairs[a]) & set(compatible_pairs[b])]
offset = -n_qubits / 2 + PENALTY * len(overlaps) / 4
h = np.full(n_qubits, 0.5)
J = np.zeros((n_qubits, n_qubits))
for a, b in overlaps:
    h[a] -= PENALTY / 4
    h[b] -= PENALTY / 4
    J[a, b] = J[b, a] = PENALTY / 4
edge_others = [(a, b, np.array([k for k in range(n_qubits) if k not in (a, b)]))
               for a, b in overlaps]

gamma, beta = ParameterVector("gamma", DEPTH), ParameterVector("beta", DEPTH)
parameter_order = list(gamma) + list(beta)
parameter_names = [str(parameter) for parameter in parameter_order]
qaoa_circuit = QuantumCircuit(n_qubits)
qaoa_circuit.h(range(n_qubits))
for layer in range(DEPTH):
    for a in range(n_qubits):
        qaoa_circuit.rz(2 * gamma[layer] * h[a], a)
    for a, b in overlaps:
        qaoa_circuit.rzz(2 * gamma[layer] * PENALTY / 4, a, b)
    for a in range(n_qubits):
        qaoa_circuit.rx(2 * beta[layer], a)

simulator = AerSimulator(method="statevector", device="CPU", precision="double", max_parallel_threads=2)
feasible_masks = [int(mask) for mask in schedules["mask"]]
feasible_energies = schedules["periods"].to_numpy(dtype=float) - len(labels)  # QUBO on matchings.
energy_order = np.argsort(feasible_energies, kind="stable")
optimal_rows = schedules["periods"].eq(5).to_numpy()

In [8]:
# Ozaeta, van Dam & McMahon, Eq. (13): https://arxiv.org/html/2012.03421v3

def energy_analytical(angles):
    g, b = angles
    sin_2b = np.sin(2 * b)
    cos_fields = np.cos(2 * g * h)
    cos_couplings = np.cos(2 * g * J)
    z_mean = sin_2b * np.sin(2 * g * h) * np.prod(cos_couplings, axis=1)
    value = float(offset + np.dot(h, z_mean))
    for i, j, others in edge_others:
        mixed = 0.5 * np.sin(4 * b) * np.sin(2 * g * J[i, j]) * (
            cos_fields[i] * np.prod(cos_couplings[i, others])
            + cos_fields[j] * np.prod(cos_couplings[j, others])
        )
        difference = np.cos(2 * g * (h[i] - h[j])) * np.prod(
            np.cos(2 * g * (J[i, others] - J[j, others]))
        )
        total = np.cos(2 * g * (h[i] + h[j])) * np.prod(
            np.cos(2 * g * (J[i, others] + J[j, others]))
        )
        value += J[i, j] * (mixed + 0.5 * sin_2b**2 * (difference - total))
    return float(value)

## 4. Select a start and train
Screen a seeded, shifted 31×31 grid over gamma `[0, 2π)` and beta `[0, π)` by
analytical mean energy; evaluate eight separated candidates with the actual CVaR
objective, then refine one start.

Normalize by feasibility, then average the lowest-energy 10% of that distribution, including fractional boundary mass. Require feasibility ≥1%. Numerical CVaR ties prefer higher conditional five-period probability, then higher feasibility.

The calculation follows the [CVaR definition of Barkoutsos et al.](https://quantum-journal.org/papers/q-2020-04-20-256/);
probabilities use Aer's [save_amplitudes_squared](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.library.save_amplitudes_squared.html).

In [9]:
# helpers

def evaluate_angles(angles):
    global simulator_calls
    key = tuple(np.asarray(angles, dtype=float).tolist())
    if key in evaluation_cache:
        return evaluation_cache[key]
    if simulator_calls >= MAX_SIMULATOR_CALLS:
        raise StopIteration("Expensive simulator-call budget reached.")
    bound = probability_circuit.assign_parameters(dict(zip(parameter_order, key)))
    simulator_calls += 1
    result = simulator.run(bound, shots=1).result()  # Unmeasured: exact probabilities.
    if not result.success:
        raise RuntimeError(f"Aer probability evaluation failed: {result.status}")
    probabilities = np.maximum(np.asarray(result.data(0)["feasible"], dtype=float), 0.0)
    feasibility = float(probabilities.sum())
    if feasibility > 1e-15:
        weights = probabilities[energy_order] / feasibility
        mass_before = np.cumsum(weights) - weights
        included = np.minimum(weights, np.maximum(CVAR_FRACTION - mass_before, 0.0))
        cvar = float(np.dot(included, feasible_energies[energy_order]) / CVAR_FRACTION)
        conditional_success = float(probabilities[optimal_rows].sum() / feasibility)
    else:
        cvar, conditional_success = 0.0, 0.0  # Ineligible; avoid dividing by zero.
    record = {"angles": list(key), "conditional_cvar": cvar,
              "feasibility_probability": feasibility,
              "conditional_five_period_probability": conditional_success}
    evaluation_cache[key] = record
    return record


def best_eligible(records):
    eligible = [row for row in records if row["feasibility_probability"] >= MIN_FEASIBILITY]
    if not eligible:
        raise RuntimeError("No eligible candidate: the fixed shortlist/search found no point with feasibility >= 0.01.")
    lowest_cvar = min(row["conditional_cvar"] for row in eligible)
    tied = [row for row in eligible if row["conditional_cvar"] <= lowest_cvar + CVAR_TIE_TOL]
    return max(tied, key=lambda row: (row["conditional_five_period_probability"],
                                      row["feasibility_probability"]))


def training_objective(angles):
    global objective_calls
    if objective_calls >= MAX_OPT_EVALS:
        raise StopIteration("COBYLA objective-evaluation budget reached.")
    objective_calls += 1
    return evaluate_angles(angles)["conditional_cvar"]


def feasibility_constraint(angles):
    return evaluate_angles(angles)["feasibility_probability"] - MIN_FEASIBILITY

In [ ]:
if RUN_TRAINING:

    training_started = perf_counter()
    rng = np.random.default_rng(SEED)

    domains = np.array([2 * np.pi, np.pi])
    steps = domains / GRID_SIZE
    axes = [(np.arange(GRID_SIZE) + rng.random()) * step for step in steps]
    grid = np.array(list(product(*axes)))
    grid_energies = np.array([energy_analytical(angles) for angles in grid])
    shortlist = []

    for index in np.argsort(grid_energies, kind="stable"):
        angles = grid[index]
        if all(np.linalg.norm(np.minimum(abs(angles - other), domains - abs(angles - other)) / steps)
               >= GRID_SEPARATION for other in shortlist):
            shortlist.append(angles)
        if len(shortlist) == SHORTLIST_SIZE:
            break

    probability_circuit = qaoa_circuit.copy()
    save_amplitudes_squared(probability_circuit, feasible_masks, label="feasible")
    probability_circuit = transpile(probability_circuit, simulator, optimization_level=1, seed_transpiler=SEED)
    evaluation_cache = {}
    simulator_calls = objective_calls = 0
    start = best_eligible([evaluate_angles(angles) for angles in shortlist])

    try:
        optimum = minimize(
            training_objective, np.array(start["angles"]), method="COBYLA",
            constraints=[{"type": "ineq", "fun": feasibility_constraint}],
            options={"maxiter": MAX_OPT_EVALS, "rhobeg": RHOBEG, "tol": OPT_TOL},
        )
        stopping_reason = str(optimum.message)

    except StopIteration as error:
        stopping_reason = str(error)

    selected = best_eligible(evaluation_cache.values())
    training_elapsed_s = perf_counter() - training_started
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

    run = {
        "run_id": run_id, "objective": "feasibility_conditioned_lower_tail_cvar",
        "settings": {
            "seed": SEED, "depth": DEPTH, "penalty": PENALTY,
            "minimum_feasibility": MIN_FEASIBILITY, "cvar_fraction": CVAR_FRACTION,
            "grid_size": GRID_SIZE, "shortlist_size": SHORTLIST_SIZE,
            "grid_separation_cells": GRID_SEPARATION, "cvar_tie_tolerance": CVAR_TIE_TOL,
            "max_optimizer_evaluations": MAX_OPT_EVALS, "max_simulator_calls": MAX_SIMULATOR_CALLS,
            "rhobeg": RHOBEG, "tol": OPT_TOL, "final_shots": FINAL_SHOTS,
            "batch_shots": BATCH_SHOTS, "final_stream": FINAL_STREAM, "chunk_size": CHUNK_SIZE,
            "detour_tie_tolerance_s": DETOUR_TIE_TOL_S,
        },
        "parameter_names": parameter_names, "compatible_pairs": compatible_pairs,
        "initial_angles": start["angles"], "selected_angles": selected["angles"],
        "conditional_cvar": selected["conditional_cvar"],
        "feasibility_probability": selected["feasibility_probability"],
        "conditional_five_period_probability": selected["conditional_five_period_probability"],
        "predicted_filtered_chunk_probability": 1 - (1 - selected["conditional_five_period_probability"]) ** CHUNK_SIZE,
        "simulator_calls": simulator_calls, "optimizer_evaluations": objective_calls,
        "stopping_reason": stopping_reason, "training_elapsed_s": training_elapsed_s,
        "raw_sample_file": f"samples_{run_id}.csv", "sampled_shots": 0, "sampling_elapsed_s": 0.0,
        "sampling_seed_rule": "SeedSequence([seed, final_stream, batch_index]).generate_state(1)[0]",
        "package_versions": {name: version(name) for name in ["qiskit", "qiskit-aer", "numpy", "scipy", "osmnx"]},
    }

    PARAMETER_FILE = OUTPUT_DIR / f"parameters_{run_id}.json"

    with PARAMETER_FILE.open("x", encoding="utf-8") as file:
        json.dump(run, file, indent=2, allow_nan=False)

    print("Selected parameters saved:", PARAMETER_FILE)
else:
    print("Training disabled.")

Selected parameters saved: /content/drive/MyDrive/quantum_road_scheduling/demo_results/parameters_20261010T184153479459Z.json


## 5. Freeze parameters and draw one ordered raw pool
Enable sampling separately.

The saved settings fix the 3,200,000-shot budget; 16 batches of 200,000 use distinct seeds, separate from initialization.

Each batch appends draw-order memory to one CSV. Completed batches can resume from the same JSON; incomplete writes stop.

In [10]:
# Restart point, use saved filename

PARAMETER_FILE = '/content/drive/MyDrive/quantum_road_scheduling/demo_results/parameters_20261010T184153479459Z.json'

In [11]:
if RUN_SAMPLING:
    if PARAMETER_FILE is None or not Path(PARAMETER_FILE).exists():
        raise FileNotFoundError("Select the saved parameter JSON before sampling.")
    PARAMETER_FILE = Path(PARAMETER_FILE)

    run = json.loads(PARAMETER_FILE.read_text(encoding="utf-8"))
    frozen = run["settings"]
    raw_file = PARAMETER_FILE.parent / run["raw_sample_file"]
    completed = int(run["sampled_shots"])
    actual_rows = 0

    if raw_file.exists():
        with raw_file.open() as file:
            actual_rows = max(0, sum(1 for _ in file) - 1)

    if actual_rows != completed:
        raise RuntimeError("The raw CSV and saved batch count disagree; recover the incomplete write before sampling.")

    total_shots, batch_shots = frozen["final_shots"], frozen["batch_shots"]

    if completed < total_shots:
        started = perf_counter()
        previous_elapsed = run["sampling_elapsed_s"]
        measured = qaoa_circuit.copy()
        measured.measure_all()
        measured = transpile(measured, simulator, optimization_level=1, seed_transpiler=frozen["seed"])
        measured = measured.assign_parameters(dict(zip(parameter_order, run["selected_angles"])))

        for batch_index in range(completed // batch_shots, ceil(total_shots / batch_shots)):
            shots = min(batch_shots, total_shots - completed)
            batch_seed = int(np.random.SeedSequence(
                [frozen["seed"], frozen["final_stream"], batch_index]
            ).generate_state(1)[0])
            result = simulator.run(measured, shots=shots, memory=True, seed_simulator=batch_seed).result()
            if not result.success:
                raise RuntimeError(f"Aer final sampling failed: {result.status}")
            masks = np.fromiter((int(bits, 2) for bits in result.get_memory(0)), dtype=np.int64, count=shots)

            pd.DataFrame({"mask": masks}).to_csv(raw_file, mode="a", header=(completed == 0), index=False)
            completed += shots

            run["sampled_shots"] = completed
            run["sampling_elapsed_s"] = previous_elapsed + perf_counter() - started

            PARAMETER_FILE.write_text(json.dumps(run, indent=2, allow_nan=False), encoding="utf-8")
    print("Raw sample pool:", raw_file)
else:
    print("Final sampling disabled.")

Raw sample pool: /content/drive/MyDrive/quantum_road_scheduling/demo_results/samples_20261010T184153479459Z.csv


## 6. Filter, rank, and report
Reject invalid masks, keeping every accepted draw in order across batch boundaries.

Use groups of 16 accepted samples.

Compare with uniform draws from all 1,216 feasible schedules. Rank sampled
schedules by periods, then cumulative route time.

In [12]:
if PARAMETER_FILE is None:
    print("Select the saved parameter JSON to report a completed pool.")
else:
    run = json.loads(Path(PARAMETER_FILE).read_text(encoding="utf-8"))
    raw_file = Path(PARAMETER_FILE).parent / run["raw_sample_file"]

    if not raw_file.exists() or run["sampled_shots"] != run["settings"]["final_shots"]:
        print("The fixed raw-shot pool is not complete yet.")

    else:
        raw = pd.read_csv(raw_file, dtype={"mask": "int64"})["mask"]
        lookup = schedules.set_index("mask")
        periods = raw.map(lookup["periods"])
        accepted_masks = raw[periods.notna()]
        accepted_periods = periods.dropna().to_numpy(dtype=int)
        raw_shots, accepted = len(raw), len(accepted_periods)
        chunk_size = run["settings"]["chunk_size"]
        chunks = accepted // chunk_size
        q_observed = float((accepted_periods == 5).mean()) if accepted else None
        observed, ci_low, ci_high = None, None, None

        if chunks:
            successes = (accepted_periods[:chunks * chunk_size] == 5).reshape(chunks, chunk_size).any(axis=1)
            observed = float(successes.mean())
            interval = binomtest(int(successes.sum()), chunks).proportion_ci(confidence_level=0.95, method="exact")
            ci_low, ci_high = float(interval.low), float(interval.high)
        reference = 1 - (1 - float(schedules["periods"].eq(5).mean())) ** chunk_size

        finalists = pd.DataFrame(columns=["mask", "periods", "batches", "total_route_time_s", "observed_samples"])
        if accepted:
            counts = accepted_masks.value_counts()
            sampled = lookup.loc[counts.index].copy()
            sampled["observed_samples"] = counts
            sampled = sampled[sampled["periods"].eq(sampled["periods"].min())]
            best_time = sampled["total_route_time_s"].min()
            finalists = sampled.loc[
                (sampled["total_route_time_s"] - best_time).abs() <= run["settings"]["detour_tie_tolerance_s"]
            ].sort_index().reset_index()

        summary = {
            "raw_shots": raw_shots, "accepted_samples": accepted,
            "acceptance_rate": accepted / raw_shots, "complete_chunks": chunks,
            "five_period_frequency_accepted": q_observed,
            "filtered_chunk_success": observed, "ci95_low": ci_low, "ci95_high": ci_high,
            "exact_predicted_filtered_success": run["predicted_filtered_chunk_probability"],
            "uniform_feasible_reference": reference,
            "training_runtime_s": run["training_elapsed_s"],
            "sampling_runtime_s": run["sampling_elapsed_s"],
            "best_sampled_masks": json.dumps([int(mask) for mask in finalists["mask"]]),
        }

        pd.DataFrame([summary]).to_csv(Path(PARAMETER_FILE).parent / f"summary_{run['run_id']}.csv", index=False)
        interval_text = f"{observed:.3%} [{ci_low:.3%}, {ci_high:.3%}]" if chunks else "Unavailable: fewer than 16 accepted samples"

        display(pd.DataFrame([
            ("Raw shots", f"{raw_shots:,}"), ("Accepted samples", f"{accepted:,}"),
            ("Acceptance rate", f"{accepted / raw_shots:.3%}"), ("Complete chunks", f"{chunks:,}"),
            ("Five-period frequency among accepted", f"{q_observed:.3%}" if accepted else "Unavailable"),
            ("Observed filtered success [exact 95% CI]", interval_text),
            ("Exact circuit-predicted filtered success", f"{run['predicted_filtered_chunk_probability']:.3%}"),
            ("Uniform-feasible reference", f"{reference:.7%}"),
            ("Training runtime", f"{run['training_elapsed_s'] / 60:.2f} min"),
            ("Sampling runtime", f"{run['sampling_elapsed_s'] / 60:.2f} min"),
        ], columns=["Measure", "Result"]).set_index("Measure"))

        finalists["cumulative_route_time_min"] = finalists["total_route_time_s"] / 60

        display(finalists[["batches", "periods", "cumulative_route_time_min", "observed_samples"]].rename(
            columns={"batches": "Best sampled schedule"}
        ).round({"cumulative_route_time_min": 6}))

,Result
Measure,
Raw shots,"3,200,000"
Accepted samples,"32,216"
Acceptance rate,1.007%
Complete chunks,"2,013"
Five-period frequency among accepted,5.919%
Observed filtered success [exact 95% CI],"63.388% [61.240%, 65.496%]"
Exact circuit-predicted filtered success,62.877%
Uniform-feasible reference,49.6177444%
Training runtime,28.49 min


,Best sampled schedule,periods,cumulative_route_time_min,observed_samples
0,BJ CF DE GH IK,5,69.063183,41
1,BE CF DJ GH IK,5,69.063183,42
2,BD CF EJ GH IK,5,69.063183,35
